In [1]:
import requests
import time
import pandas as pd
from datetime import datetime, timedelta

In [2]:
API_KEY = "qrZUHabbbGHJWU95326BITrpe1ZX6SbC79MvFmbIKuEICM9l"

MAX_MINUTE_RETRIES = 3   # retry attempts for minute limit
MINUTE_SLEEP = 60        # NYT allows ~5 requests per minute

def fetch_articles(query, begin_date, end_date, page):

    url = "https://api.nytimes.com/svc/search/v2/articlesearch.json"

    params = {
        "api-key": API_KEY,
        "q": query,
        "begin_date": begin_date,
        "end_date": end_date,
        "page": page,
    }

    retries = 0

    while retries <= MAX_MINUTE_RETRIES:
        response = requests.get(url, params=params)

        if response.status_code == 200:
            data = response.json()
            return data.get("response", {}).get("docs", [])

        if response.status_code == 429:
            retries += 1

            if retries > MAX_MINUTE_RETRIES:
                print("Daily rate limit likely reached.")
                return None  # signal to stop everything

            print(f"Minute rate limit hit. Sleeping {MINUTE_SLEEP}s...")
            time.sleep(MINUTE_SLEEP)
            continue
    
        print("Error:", response.status_code)
        return []

    return None


def get_all_articles(query, start_date, end_date):

    all_articles = []
    current_date = start_date
    stop_collection = False

    while current_date <= end_date and not stop_collection:

        month_end = (current_date.replace(day=28) + timedelta(days=4)).replace(day=1) - timedelta(days=1)
        if month_end > end_date:
            month_end = end_date

        begin_str = current_date.strftime("%Y%m%d")
        end_str = month_end.strftime("%Y%m%d")

        print(f"Fetching {begin_str} to {end_str}")

        month_articles = []          # store articles for THIS month only
        month_failed = False         # track if any request failed

        for page in range(5):

            docs = fetch_articles(query, begin_str, end_str, page)

            if docs is None:
                month_failed = True  # mark month as incomplete
                stop_collection = True
                break

            if not docs:
                break

            month_articles.extend(docs)
            
            time.sleep(12)

        # Only commit the month if ALL requests succeeded
        if not month_failed:
            all_articles.extend(month_articles)
        else:
            print(f"Skipping incomplete month {begin_str}-{end_str}")

        current_date = month_end + timedelta(days=1)

    print(f"Returning {len(all_articles)} articles collected so far.")
    return pd.json_normalize(all_articles)

In [ ]:
# Run it
start = datetime(2022, 9, 1)
end = datetime(2026, 2, 28)

df = get_all_articles("economy", start, end)

df.head()

Fetching 20220901 to 20220930
Fetching 20221001 to 20221031
Fetching 20221101 to 20221130
Fetching 20221201 to 20221231
Fetching 20230101 to 20230131
Fetching 20230201 to 20230228
Fetching 20230301 to 20230331
Fetching 20230401 to 20230430
Fetching 20230501 to 20230531
Fetching 20230601 to 20230630
Fetching 20230701 to 20230731
Fetching 20230801 to 20230831
Fetching 20230901 to 20230930
Fetching 20231001 to 20231031
Fetching 20231101 to 20231130
Fetching 20231201 to 20231231
Fetching 20240101 to 20240131
Fetching 20240201 to 20240229
Fetching 20240301 to 20240331
Fetching 20240401 to 20240430
Fetching 20240501 to 20240531
Fetching 20240601 to 20240630
Fetching 20240701 to 20240731
Fetching 20240801 to 20240831
Fetching 20240901 to 20240930
Fetching 20241001 to 20241031
Fetching 20241101 to 20241130
Fetching 20241201 to 20241231
Fetching 20250101 to 20250131
Fetching 20250201 to 20250228
Fetching 20250301 to 20250331
Fetching 20250401 to 20250430
Fetching 20250501 to 20250531
Fetching 2

In [ ]:
df.to_csv("nytimes_economy_3.csv", index=False)